# Pyro Model for ODMR Spectra Fitting

This notebook builds a Pyro model to fit ODMR spectra using the `NV_ODMR` model from `odmrsimulator.py`.
The model optimizes the following parameters:
- E_rad_s: strain parameter (rad/s)
- D0: zero-field splitting at T0 (MHz)
- alpha: temperature coefficient for D (MHz/K)
- T1_0: T1 at T0 (us)
- beta: T1 temperature coefficient (1/K)
- gamma: T2 temperature coefficient (1/K)
- T2_0: T2 at T0 (us)
- rabi_rate: Rabi frequency (MHz)

In [ ]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS, HMC
from pyro.infer.autoguide import init_to_value
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import importlib
from sklearn.preprocessing import MinMaxScaler

# Import the ODMR simulator
import odmrsimulator
import helper
importlib.reload(odmrsimulator)

torch.set_default_dtype(torch.float64)

## Define Pyro Model

The model samples parameters from priors and uses NV_ODMR to generate predictions.

In [ ]:
def model(data):
    """
    Pyro model for ODMR spectra fitting.
    
    Parameters:
    -----------
    data : tuple
        (RF_freq, T, BNV, y_obs) where:
        - RF_freq: array of RF frequencies (MHz)
        - T: temperature (K)
        - BNV: magnetic field (Gauss)
        - y_obs: observed PL values
    """
    # Unpack data tuple
    if len(data) != 4:
        raise ValueError(
            f"Data tuple must have 4 elements (RF_freq, T, BNV, y_obs), got {len(data)}"
        )
    
    RF_freq, T, BNV, y_obs = data
    
    # Check if y_obs is accidentally a nested tuple (common mistake)
    # If so, extract the actual y_obs and warn the user
    if isinstance(y_obs, tuple) and len(y_obs) == 2:
        import warnings
        warnings.warn(
            f"y_obs appears to be a tuple of length 2. "
            f"This suggests the data tuple was incorrectly structured. "
            f"Extracting y_obs from the tuple. "
            f"Please fix the data tuple to be (RF_freq, T, BNV, y_obs) not (RF_freq, T, BNV, (x_obs, y_obs))."
        )
        # Extract the actual y_obs (assuming it's the second element, which should be the PL values)
        # The first element would be x_obs (frequencies), second is y_obs (PL values)
        y_obs = y_obs[1]
    
    # Ensure all inputs are properly shaped tensors
    if isinstance(RF_freq, torch.Tensor):
        RF_freq = RF_freq.squeeze()
        if RF_freq.ndim == 0:
            RF_freq = RF_freq.unsqueeze(0)
    else:
        RF_freq = torch.tensor(RF_freq, dtype=torch.float64).squeeze()
        if RF_freq.ndim == 0:
            RF_freq = RF_freq.unsqueeze(0)
    
    # Ensure y_obs is a 1D tensor
    # Handle different input types (tensor, numpy array, list, tuple, etc.)
    if isinstance(y_obs, torch.Tensor):
        y_obs = y_obs.squeeze()
        # Ensure it's the right dtype
        if y_obs.dtype != torch.float64:
            y_obs = y_obs.double()
    elif isinstance(y_obs, np.ndarray):
        y_obs = torch.tensor(y_obs, dtype=torch.float64).squeeze()
    elif isinstance(y_obs, (list, tuple)):
        # If it's a tuple/list of tensors/arrays, handle appropriately
        if len(y_obs) > 0 and isinstance(y_obs[0], torch.Tensor):
            # List/tuple of tensors - this shouldn't happen, but handle it
            # Take the first element if it's a list of tensors
            y_obs = y_obs[0] if len(y_obs) == 1 else y_obs[-1]  # Take last element
            if isinstance(y_obs, torch.Tensor):
                y_obs = y_obs.squeeze().double()
            else:
                y_obs = torch.tensor(np.asarray(y_obs), dtype=torch.float64).squeeze()
        else:
            # Regular list/tuple of numbers - convert to tensor
            y_obs = torch.tensor(np.asarray(y_obs), dtype=torch.float64).squeeze()
    else:
        # Try to convert directly via numpy
        try:
            y_obs = torch.tensor(np.asarray(y_obs, dtype=np.float64), dtype=torch.float64).squeeze()
        except (ValueError, TypeError) as e:
            raise ValueError(
                f"Cannot convert y_obs to tensor. y_obs type: {type(y_obs)}, "
                f"y_obs repr: {repr(y_obs)[:100]}. Error: {e}"
            )
    
    # Get data size from y_obs (this is what Pyro uses for plate size)
    # Must be a Python int, not a tensor
    if y_obs.ndim == 0:
        n_data = 1
    else:
        n_data = int(y_obs.shape[0])
    
    # Ensure RF_freq matches y_obs length
    if RF_freq.shape[0] != n_data:
        # If RF_freq is scalar or wrong size, we need to handle this
        # This shouldn't happen in normal operation, but Pyro tracing might cause it
        if RF_freq.numel() == 1 and n_data > 1:
            # During tracing, Pyro might pass a scalar - we can't handle this properly
            # So we'll raise an error with a helpful message
            raise ValueError(
                f"RF_freq has shape {RF_freq.shape} but y_obs has shape {y_obs.shape}. "
                f"RF_freq must be an array with the same length as y_obs."
            )
        elif RF_freq.shape[0] != n_data:
            raise ValueError(
                f"RF_freq length ({RF_freq.shape[0]}) doesn't match y_obs length ({n_data})"
            )
    
    # Sample parameters from priors
    E_rad_s = pyro.sample("E_rad_s", dist.Normal(3.14 * 4 * 50e6, 1e7)).double()
    #D0 = pyro.sample("D0", dist.Normal(2877.0, 10.0)).double()
    #alpha = pyro.sample("alpha", dist.Normal(-0.72, 0.1)).double()
    #T1_0 = pyro.sample("T1_0", dist.LogNormal(np.log(1e6), 0.5)).double()
    #beta = pyro.sample("beta", dist.Normal(0.01, 0.005)).double()
    #gamma = pyro.sample("gamma", dist.Normal(0.005, 0.002)).double()
    #T2_0 = pyro.sample("T2_0", dist.LogNormal(np.log(1e3), 0.5)).double()
    rabi_rate = pyro.sample("rabi_rate", dist.Normal(1.5, 0.5)).double()
    
    # Sample noise variance
    var = pyro.sample("var", dist.HalfNormal(scale=0.1)).double()
    
    # Generate predictions using NV_ODMR
    PL_pred = np.zeros(len(RF_freq))
    for i, RF_freq in enumerate(RF_freq):
        PL_pred[i] = odmrsimulator.NV_ODMR(BNV, RF_freq, rabi_rate, E_rad_s_param=E_rad_s)

    
    # Ensure PL_pred is 1-D and matches y_obs shape exactly     
    PL_pred = PL_pred.squeeze()
    if PL_pred.ndim == 0:
        PL_pred = PL_pred.unsqueeze(0)
    
    # Ensure shapes match
    if PL_pred.shape[0] != n_data:
        raise ValueError(
            f"PL_pred has shape {PL_pred.shape} but y_obs has shape {y_obs.shape}. "
            f"PL_pred length ({PL_pred.shape[0]}) must match y_obs length ({n_data})"
        )
    
    # Sample observations - use Independent to handle vectorized observations
    with pyro.plate("data", n_data):
        pyro.sample("obs", dist.Normal(PL_pred, var), obs=y_obs)
    
    return PL_pred

In [ ]:
## import data file
fpath = './saved_data/cycle1'

df_= pd.read_csv(fpath, sep=',', header = 0); 
df= df_.iloc[0:, 1:-1]
df.drop(columns= ['25 C-lower power', '15', '10', '10.1', '-30', '-20'], inplace= True)
df.iloc[:, 2:].plot(legend= False); plt.show()

# define and scale the frequency axis 
x_esr = df.frequency.values
y_esr = df.iloc[:, 1:7]


torch.set_default_dtype(torch.float64)

y_esr = y_esr.apply(lambda x: x - x[:10].mean())#+0.01
# y_esr = -1*y_esr
y_esr = y_esr.apply(lambda x: x/x.max())
plt.plot(x_esr,y_esr);

In [ ]:
x_obs = torch.tensor(x_esr, dtype=torch.float64)
y_obs = torch.tensor(y_esr.iloc[:,0].values, dtype=torch.float64)

## Verify Data Structure Before MCMC

**CRITICAL**: Make sure your data tuple is structured correctly as `(RF_freq, T, BNV, y_obs)` - NOT nested!

In [ ]:
# this will be the data we'll start with, shown as red circles
def dataslicer(x, y, col1 =0 ,col2=1):
    x_scale_tensor = torch.tensor(x_esr).double()
    # squeeze the selected column to produce a 1-D tensor (N,) instead of (N,1)
    y_vals = y_esr.iloc[:, col1:col2].values.squeeze()
    y_scale_tensor = torch.tensor(y_vals).double()
    return x_scale_tensor, y_scale_tensor

In [ ]:
x_obs, y_obs = dataslicer(x_esr, y_esr, col1=0, col2=1)
T_ = 273.15 + 25.0  # Example temperature in Kelvin

data_ = (test_RF_freq, T_, test_BNV, (x_obs.clone().detach().double(), y_obs.clone().detach().double()))
data = data_[3]

In [ ]:
# Prepare data for MCMC
# The model expects: (RF_freq, T, BNV, y_obs)
# RF_freq: array of RF frequencies (MHz) - should match length of y_obs
# T: temperature (K) - scalar
# BNV: magnetic field (Gauss) - scalar  
# y_obs: observed PL values - 1D array matching RF_freq length

# Example with experimental data (if you have it):
# Assuming you have x_obs (frequencies) and y_obs (PL values) from your data
# x_obs and y_obs should have the same length

# Convert to tensors
RF_freq_tensor = torch.tensor(test_RF_freq, dtype=torch.float64)  # or use your frequency array
T_tensor = torch.tensor(300.0, dtype=torch.float64)  # temperature in Kelvin
BNV_tensor = torch.tensor(0.0, dtype=torch.float64)  # magnetic field in Gauss
y_obs_tensor = y_obs  # PL values

# Ensure y_obs is 1D
if y_obs_tensor.ndim > 1:
    y_obs_tensor = y_obs_tensor.flatten()

# Ensure RF_freq matches y_obs length
if RF_freq_tensor.shape[0] != y_obs_tensor.shape[0]:
    raise ValueError(f"RF_freq length ({RF_freq_tensor.shape[0]}) must match y_obs length ({y_obs_tensor.shape[0]})")

# Create data tuple - CORRECT FORMAT: (RF_freq, T, BNV, y_obs)
data = (RF_freq_tensor, T_tensor, BNV_tensor, y_obs_tensor)

print(f"Data tuple prepared:")
print(f"  RF_freq shape: {data[0].shape}")
print(f"  T (scalar): {data[1].item()}")
print(f"  BNV (scalar): {data[2].item()}")
print(f"  y_obs shape: {data[3].shape}")
print(f"\nAll shapes match: {data[0].shape[0] == data[3].shape[0]}")

## Run MCMC Inference

Use NUTS sampler for Bayesian inference.

In [ ]:
# # Set initial values
# init_vals = {
#     "E_rad_s": torch.tensor(test_E_rad_s),
#     "D0": torch.tensor(test_D0),
#     "alpha": torch.tensor(test_alpha),
#     "T1_0": torch.tensor(test_T1_0),
#     "beta": torch.tensor(test_beta),
#     "gamma": torch.tensor(test_gamma),
#     "T2_0": torch.tensor(test_T2_0),
#     "rabi_rate": torch.tensor(test_rabi_rate),
#     "var": torch.tensor(noise_std**2),
# }


init_vals = {
    "E_rad_s": torch.tensor(test_E_rad_s),
    "D0": torch.tensor(test_D0),
    "alpha": torch.tensor(test_alpha),
    "T1_0": torch.tensor(test_T1_0),
    "beta": torch.tensor(test_beta),
    "gamma": torch.tensor(test_gamma),
    "T2_0": torch.tensor(test_T2_0),
    "rabi_rate": torch.tensor(test_rabi_rate),
    "var": torch.tensor(noise_std**2),
}


# Create NUTS kernel
kernel = NUTS(
    model, 
    jit_compile=False,  # Set to False due to numpy/qutip dependencies
    init_strategy=init_to_value(values=init_vals),
    max_tree_depth=3,
    ignore_jit_warnings=True
)

# Run MCMC
mcmc = MCMC(kernel, num_samples=2, warmup_steps=2, num_chains=1)
mcmc.run(data_)

# Get samples
samples = mcmc.get_samples()

## Analyze Results

Extract posterior statistics and visualize results.

In [ ]:
# Print posterior statistics
print("Posterior Statistics:")
print("=" * 50)
for param_name in ["E_rad_s", "D0", "alpha", "T1_0", "beta", "gamma", "T2_0", "rabi_rate"]:
    param_samples = samples[param_name].detach().cpu().numpy()
    mean_val = param_samples.mean()
    std_val = param_samples.std()
    print(f"{param_name:12s}: {mean_val:12.4e} ± {std_val:12.4e}")

# # Compare with true values
# print("True Values:")
# print("=" * 50)
# true_vals = {
#     "E_rad_s": test_E_rad_s,
#     "D0": D0_true,
#     "alpha": alpha_true,
#     "T1_0": T1_0_true,
#     "beta": beta_true,
#     "gamma": gamma_true,
#     "T2_0": T2_0_true,
#     "rabi_rate": rabi_rate_true,
# }
# for param_name, true_val in true_vals.items():
#     print(f"{param_name:12s}: {true_val:12.4e}")

In [ ]:
# Generate predictions using posterior mean parameters
posterior_means = {}
for param_name in ["E_rad_s", "D0", "alpha", "T1_0", "beta", "gamma", "T2_0", "rabi_rate"]:
    posterior_means[param_name] = samples[param_name].mean().item()


T_true = T_tensor.item()
BNV_true = BNV_tensor.item()


PL_pred_mean = NV_ODMR_wrapper(
    test_RF_freq, T_true, BNV_true, posterior_means["rabi_rate"],
    posterior_means["E_rad_s"], posterior_means["D0"], posterior_means["alpha"],
    posterior_means["T1_0"], posterior_means["beta"], posterior_means["gamma"],
    posterior_means["T2_0"]
)

# Plot results
plt.figure(figsize=(12, 5))

# Plot 1: Data and fit
plt.subplot(1, 2, 1)
# plt.plot(test_RF_freq, PL_pred_mean.numpy(), 'b-', label='True PL', linewidth=2)
# plt.plot(test_RF_freq, y_obs.numpy(), 'r.', label='Observed PL', alpha=0.5, markersize=3)
plt.plot(test_RF_freq, (PL_pred_mean.numpy())*1, 'g--', label='Posterior Mean', linewidth=2)
plt.xlabel('RF Frequency (MHz)')
plt.ylabel('PL (a.u.)')
plt.title('ODMR Spectra Fit')
plt.legend()
plt.grid(alpha=0.3)

# # Plot 2: Residuals
# plt.subplot(1, 2, 2)
# residuals = (y_obs + PL_pred_mean).numpy()
# plt.plot(test_RF_freq, residuals, 'k.', alpha=0.5, markersize=3)
# plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
# plt.xlabel('RF Frequency (MHz)')
# plt.ylabel('Residuals')
# plt.title('Residuals')
# plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot posterior distributions
param_names = ["E_rad_s", "D0", "alpha", "T1_0", "beta", "gamma", "T2_0", "rabi_rate"]
n_params = len(param_names)
n_cols = 4
n_rows = (n_params + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten() if n_params > 1 else [axes]

for i, param_name in enumerate(param_names):
    param_samples = samples[param_name].detach().cpu().numpy()
    axes[i].hist(param_samples, bins=50, alpha=0.7, edgecolor='black')
    axes[i].axvline(true_vals[param_name], color='r', linestyle='--', linewidth=2, label='True')
    axes[i].axvline(param_samples.mean(), color='g', linestyle='--', linewidth=2, label='Posterior Mean')
    axes[i].set_xlabel(param_name)
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(alpha=0.3)

# Remove extra subplots
for i in range(n_params, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Also test direct call for comparison
odmrsimulator.D0 = test_D0
odmrsimulator.T0 = 300.0
odmrsimulator.alpha = test_alpha
odmrsimulator.T1_0 = test_T1_0
odmrsimulator.beta = test_beta
odmrsimulator.T2_0 = test_T2_0
odmrsimulator.gamma = test_gamma
odmrsimulator.E_rad_s = test_E_rad_s
odmrsimulator.B = test_BNV

PL_direct = np.zeros(len(test_RF_freq))
for i, RF_freq in enumerate(test_RF_freq):
    PL_direct[i] = odmrsimulator.NV_ODMR(test_BNV, RF_freq, test_rabi_rate, test_T, E_rad_s_param=test_E_rad_s)

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(test_RF_freq, PL_test.numpy(), 'b-', label='Wrapper', linewidth=2)
plt.plot(test_RF_freq, PL_direct, 'r--', label='Direct call', linewidth=2, alpha=0.7)
plt.xlabel('RF Frequency (MHz)')
plt.ylabel('PL (a.u.)')
plt.title('Wrapper vs Direct Call')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
if len(PL_test.numpy()) == len(PL_direct):
    diff = PL_test.numpy() - PL_direct
    plt.plot(test_RF_freq, diff, 'k-', linewidth=1)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=1)
    plt.xlabel('RF Frequency (MHz)')
    plt.ylabel('Difference')
    plt.title('Difference (Wrapper - Direct)')
    plt.grid(alpha=0.3)
    print(f"Max difference: {np.max(np.abs(diff)):.2e}")
    print(f"Mean difference: {np.mean(np.abs(diff)):.2e}")

plt.tight_layout()
plt.show()

# Check if we get the expected two dips
print(f"PL range: [{PL_test.min().item():.4f}, {PL_test.max().item():.4f}]")
print(f"PL variation: {PL_test.max().item() - PL_test.min().item():.4f}")